# 💰 CAIEP L1 — AI Personal Finance Tracker
## Noob Dev | Powered by **Ollama** 


---

## ⚙️ Cell 1 — Imports & Model Configuration



In [122]:
import sqlite3
import json
import datetime
import gradio as gr
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)

MODEL = "llama3:latest"
DB_PATH = "finance.db"

print("✅ Imports done! Model:", MODEL)
print("📁 Database will be at:", DB_PATH)
DB_PATH = "finance.db"

print("✅ Imports done! Model:", MODEL)
print("📁 Database will be at:", DB_PATH)

✅ Imports done! Model: llama3:latest
📁 Database will be at: finance.db
✅ Imports done! Model: llama3:latest
📁 Database will be at: finance.db


## 🗄️ Cell 2 — Database Setup



In [123]:
def init_db():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS salary (
            id         INTEGER PRIMARY KEY AUTOINCREMENT,
            amount     REAL    NOT NULL,
            month      TEXT    NOT NULL,
            year       INTEGER NOT NULL,
            created_at TEXT    NOT NULL
        )
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS expenses (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            amount      REAL    NOT NULL,
            category    TEXT    NOT NULL,
            description TEXT,
            date        TEXT    NOT NULL,
            created_at  TEXT    NOT NULL
        )
    """)

    conn.commit()
    conn.close()
    print("✅ Database initialised:", DB_PATH)


init_db()

✅ Database initialised: finance.db


## 🔧 Cell 3 — Tool 1: `set_salary()`

Called when the user mentions their income. Deletes any existing record for the same month/year first (replace behaviour), then inserts the new one.

In [124]:
def set_salary(amount: float, month: str, year: int) -> str:
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    created_at = datetime.datetime.now().isoformat()

    cursor.execute("DELETE FROM salary WHERE month = ? AND year = ?", (month, year))
    cursor.execute(
        "INSERT INTO salary (amount, month, year, created_at) VALUES (?, ?, ?, ?)",
        (amount, month, year, created_at)
    )

    conn.commit()
    conn.close()
    return f"Salary of ${amount:,.2f} for {month} {year} saved successfully."


# Quick test
print(set_salary(3500.0, "May", 2026))

Salary of $3,500.00 for May 2026 saved successfully.


## 🔧 Cell 4 — Tool 2: `log_expense()`



In [125]:
def log_expense(amount: float, category: str, description: str = "", date: str = "") -> str:
    if not date:
        date = datetime.date.today().isoformat()

    created_at = datetime.datetime.now().isoformat()

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute(
        "INSERT INTO expenses (amount, category, description, date, created_at) VALUES (?, ?, ?, ?, ?)",
        (amount, category, description, date, created_at)
    )
    conn.commit()

    now = datetime.date.today()
    month_name = now.strftime("%B")
    year = now.year

    cursor.execute(
        "SELECT COALESCE(amount, 0) FROM salary WHERE month = ? AND year = ?",
        (month_name, year)
    )
    row = cursor.fetchone()
    salary = row[0] if row else 0.0

    cursor.execute(
        "SELECT COALESCE(SUM(amount), 0) FROM expenses WHERE strftime('%Y-%m', date) = ?",
        (now.strftime("%Y-%m"),)
    )
    total_spent = cursor.fetchone()[0]
    remaining = salary - total_spent

    conn.close()
    return (
        f"Logged ${amount:,.2f} for {category}. "
        f"Spent so far: ${total_spent:,.2f} | Remaining: ${remaining:,.2f}"
    )


print(log_expense(900.0, "Rent", "Monthly rent"))
print(log_expense(67.50, "Groceries", "Walmart"))

Logged $900.00 for Rent. Spent so far: $27,034.50 | Remaining: $-23,534.50
Logged $67.50 for Groceries. Spent so far: $27,102.00 | Remaining: $-23,602.00


## 🔧 Cell 5 — Tool 3: `get_balance()`


In [126]:
def get_balance() -> str:
    now = datetime.date.today()
    month_name = now.strftime("%B")
    year = now.year

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute(
        "SELECT COALESCE(amount, 0) FROM salary WHERE month = ? AND year = ?",
        (month_name, year)
    )
    row = cursor.fetchone()
    salary = row[0] if row else 0.0

    cursor.execute(
        "SELECT COALESCE(SUM(amount), 0) FROM expenses WHERE strftime('%Y-%m', date) = ?",
        (now.strftime("%Y-%m"),)
    )
    total_spent = cursor.fetchone()[0]
    remaining = salary - total_spent

    conn.close()

    if salary == 0:
        return "No salary recorded for this month yet. Tell me your monthly income first!"

    return (
        f"📅 {month_name} {year} Snapshot\n"
        f"💰 Salary:    ${salary:,.2f}\n"
        f"💸 Spent:     ${total_spent:,.2f}\n"
        f"✅ Remaining: ${remaining:,.2f}"
    )


print(get_balance())

📅 May 2026 Snapshot
💰 Salary:    $3,500.00
💸 Spent:     $27,102.00
✅ Remaining: $-23,602.00


## 🔧 Cell 6 — Tool 4: `get_expense_summary()`


In [127]:
def get_expense_summary() -> str:
    now = datetime.date.today()
    month_name = now.strftime("%B")
    year = now.year

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute(
        "SELECT COALESCE(amount, 0) FROM salary WHERE month = ? AND year = ?",
        (month_name, year)
    )
    row = cursor.fetchone()
    salary = row[0] if row else 0.0

    cursor.execute("""
        SELECT category, SUM(amount) as total
        FROM expenses
        WHERE strftime('%Y-%m', date) = ?
        GROUP BY category
        ORDER BY total DESC
    """, (now.strftime("%Y-%m"),))

    rows = cursor.fetchall()
    conn.close()

    if not rows:
        return "No expenses recorded for this month yet."

    total_spent = sum(r[1] for r in rows)
    remaining = salary - total_spent

    lines = [f"📊 {month_name} {year} Spending Breakdown\n"]
    for category, total in rows:
        pct = (total / salary * 100) if salary > 0 else 0.0
        lines.append(f"  • {category:<15} ${total:>8,.2f}  ({pct:.1f}% of salary)")

    lines.append(f"\n💸 Total Spent: ${total_spent:,.2f}")
    lines.append(f"✅ Remaining:   ${remaining:,.2f}")
    return "\n".join(lines)


print(get_expense_summary())

📊 May 2026 Spending Breakdown

  • Rent            $25,200.00  (720.0% of salary)
  • Groceries       $1,890.00  (54.0% of salary)
  • Food & Drink    $   12.00  (0.3% of salary)

💸 Total Spent: $27,102.00
✅ Remaining:   $-23,602.00


## 📋 Cell 7 — Tool Definitions (JSON Schema for the AI)



In [128]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "set_salary",
            "description": (
                "Call this when the user mentions their salary, income, or monthly earnings. "
                "Examples: 'My salary is $3000', 'I earn 4000 a month', 'Set my income to 2800'."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "amount": {"type": "number", "description": "Salary amount in dollars"},
                    "month":  {"type": "string", "description": "Full month name e.g. 'May'. Use current month if not specified."},
                    "year":   {"type": "integer", "description": "Four-digit year e.g. 2026. Use current year if not specified."}
                },
                "required": ["amount", "month", "year"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "log_expense",
            "description": (
                "Call this whenever the user mentions spending money or any purchase. "
                "Infer category from context: coffee→'Food & Drink', rent→'Rent', "
                "Netflix→'Subscriptions', gas→'Transport', doctor→'Healthcare'. "
                "Examples: 'Spent $12 on coffee', 'Paid rent $900', 'Netflix $15.99'."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "amount":      {"type": "number", "description": "Expense amount in dollars"},
                    "category":    {"type": "string", "description": "Category e.g. 'Food & Drink', 'Rent', 'Groceries', 'Transport', 'Subscriptions', 'Healthcare'"},
                    "description": {"type": "string", "description": "Short description of the purchase"},
                    "date":        {"type": "string", "description": "Date in YYYY-MM-DD format. Leave empty to use today."}
                },
                "required": ["amount", "category"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_balance",
            "description": (
                "Call this when the user asks about remaining budget, balance, or how much is left. "
                "Examples: 'How much do I have left?', 'What is my remaining budget?', 'Am I on track?'."
            ),
            "parameters": {"type": "object", "properties": {}, "required": []}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_expense_summary",
            "description": (
                "Call this when the user wants a spending breakdown or summary by category. "
                "Examples: 'Show my spending breakdown', 'Where am I spending the most?', 'Give me a summary'."
            ),
            "parameters": {"type": "object", "properties": {}, "required": []}
        }
    }
]

print(f"✅ {len(TOOLS)} tools defined:", [t['function']['name'] for t in TOOLS])

✅ 4 tools defined: ['set_salary', 'log_expense', 'get_balance', 'get_expense_summary']


## 🔄 Cell 8 — The Tool-Calling Agent Loop



In [129]:
def run_agent(user_message: str, history: list) -> str:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    # Replay history
    for msg in history:
        if isinstance(msg, dict):
            messages.append({"role": msg["role"], "content": msg["content"]})
        else:
            human, assistant = msg
            messages.append({"role": "user", "content": human})
            messages.append({"role": "assistant", "content": assistant})

    messages.append({"role": "user", "content": user_message})

    # Simple chat call (NO TOOLS)
    response = client.chat.completions.create(
        model="llama3:latest",
        messages=messages
    )

    return response.choices[0].message.content

## 🖥️ Cell 9 — Gradio UI



In [130]:
def chat(user_message: str, history: list):
    """Gradio event handler."""
    if not user_message.strip():
        return "", history

    try:
        reply = run_agent(user_message, history)
    except Exception as e:
        reply = f"⚠️ Error: {str(e)}\n\nMake sure Ollama is running: `ollama serve`"

    # Gradio 4+ format: append dicts with role/content
    history.append({"role": "user",      "content": user_message})
    history.append({"role": "assistant", "content": reply})
    return "", history


def clear_chat():
    return []


with gr.Blocks(
    title="💰 AI Finance Tracker",
    theme=gr.themes.Soft(primary_hue="emerald"),
    css=".gradio-container { max-width: 800px; margin: auto; } footer { display: none !important; }"
) as demo:

    gr.Markdown("""
    # 💰 AI Personal Finance Tracker
    **Powered by Ollama + llama3 — 100% Free, No API Key Needed**

    Try: *"My salary is $3,500"* · *"Paid rent $900"* · *"How much is left?"* · *"Show spending breakdown"*
    """)

    # type="messages" is required for Gradio 4+
    chatbot = gr.Chatbot(
        label="Finance Assistant",
        height=450,
    )

    with gr.Row():
        msg_box = gr.Textbox(
            placeholder="Tell me about your income or expenses...",
            label="",
            scale=4,
            autofocus=True,
        )
        send_btn = gr.Button("Send 💬", variant="primary", scale=1)

    clear_btn = gr.Button("🗑️ Clear Chat", variant="secondary")

    gr.Examples(
        examples=[
            "My monthly salary is $3,500",
            "Paid rent today — $900",
            "Spent $12 on coffee this morning",
            "Bought groceries for $67.50 at Walmart",
            "Netflix subscription renewed, $15.99",
            "Filled up my gas tank, it was $55",
            "Doctor appointment copay $30",
            "How much do I have left?",
            "Show me my spending breakdown",
        ],
        inputs=msg_box,
    )

    send_btn.click(chat,       [msg_box, chatbot], [msg_box, chatbot])
    msg_box.submit(chat,       [msg_box, chatbot], [msg_box, chatbot])
    clear_btn.click(clear_chat, [],                [chatbot])


print("✅ UI built. Run Cell 10 to launch!")

C:\Users\ACER\AppData\Local\Temp\ipykernel_36904\664314400.py:21: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


✅ UI built. Run Cell 10 to launch!


In [131]:
print("🚀 Launching AI Finance Tracker...")
print(f"📦 Model : {MODEL}  Ollama llama3")
print(f"🗄️  DB    : {DB_PATH}")
print("➡️  Open: http://127.0.0.1:7860")

demo.launch(share=False)

🚀 Launching AI Finance Tracker...
📦 Model : llama3:latest  Ollama llama3
🗄️  DB    : finance.db
➡️  Open: http://127.0.0.1:7860
* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


## 🧪 Optional Cell 11 — Direct Function Tests

Run this to verify all 4 tools work before testing in the UI.

In [132]:
now = datetime.date.today()
current_month = now.strftime("%B")
current_year  = now.year

print("=" * 50)
print("1️⃣  set_salary:")
print(set_salary(3500.0, current_month, current_year))

print("\n2️⃣  log_expense (rent):")
print(log_expense(900.0, "Rent", "Monthly rent"))

print("\n3️⃣  log_expense (groceries):")
print(log_expense(67.50, "Groceries", "Walmart"))

print("\n4️⃣  get_balance:")
print(get_balance())

print("\n5️⃣  get_expense_summary:")
print(get_expense_summary())

print("\n✅ All tests passed!")

1️⃣  set_salary:
Salary of $3,500.00 for May 2026 saved successfully.

2️⃣  log_expense (rent):
Logged $900.00 for Rent. Spent so far: $28,002.00 | Remaining: $-24,502.00

3️⃣  log_expense (groceries):
Logged $67.50 for Groceries. Spent so far: $28,069.50 | Remaining: $-24,569.50

4️⃣  get_balance:
📅 May 2026 Snapshot
💰 Salary:    $3,500.00
💸 Spent:     $28,069.50
✅ Remaining: $-24,569.50

5️⃣  get_expense_summary:
📊 May 2026 Spending Breakdown

  • Rent            $26,100.00  (745.7% of salary)
  • Groceries       $1,957.50  (55.9% of salary)
  • Food & Drink    $   12.00  (0.3% of salary)

💸 Total Spent: $28,069.50
✅ Remaining:   $-24,569.50

✅ All tests passed!
